# Notebook 04 — Groundwater Trend and Drought Analysis
**Stage 6 of the Ghana Groundwater–Subsidence Research Pipeline**

Performs Mann-Kendall trend tests, Sen's slope estimation, drought episode identification, CHIRPS rainfall comparison, and gridded pixel-wise trend mapping.

In [ ]:
import os, sys
# Ensure relative paths resolve from repo root regardless of how notebook is launched
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import matplotlib.ticker as mticker
from pathlib import Path
import pymannkendall as mk
from scipy import stats
from scipy.signal import detrend

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT     = Path('.')
OUT_TS   = ROOT / 'data' / 'interim' / 'time_series'
OUT_GW   = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
FIG_DIR  = ROOT / 'outputs' / 'figures'
TAB_DIR  = ROOT / 'outputs' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

CHIRPS_FILE = ROOT / 'data' / 'raw' / 'chirps' / 'chirps-v2.0.monthly.nc'

# Study area bbox (GRACE native degrees)
GHANA_LAT = slice(4.0, 11.5)
GHANA_LON = slice(-3.5, 1.5)

# Drought SGWA thresholds (from CLAUDE.md §8.6.3)
DROUGHT_BINS  = [-np.inf, -2.0, -1.5, -1.0, -0.5, 0.5, 1.0, np.inf]
DROUGHT_LABELS = ['Extreme drought','Severe drought','Moderate drought',
                  'Mild drought','Near normal','Mildly wet','Wet']

print('Imports OK')
print(f'CHIRPS file present: {CHIRPS_FILE.exists()}')


In [ ]:
# ── 2. Load GWSA monthly time series ──────────────────────────────────────
df = pd.read_csv(OUT_TS / 'gwsa_monthly.csv', index_col=0, parse_dates=True)
df.index = df.index.to_period('M').to_timestamp(how='start')
df = df[~df.index.duplicated(keep='first')].sort_index()

print(f'GWSA series: {len(df)} months  '
      f'({df.index[0].strftime("%Y-%m")} → {df.index[-1].strftime("%Y-%m")})')
print(df[['gwsa_mm','sgwa','drought_class']].tail(6).to_string())


In [ ]:
# ── 3. Mann-Kendall trend tests ────────────────────────────────────────────
series = df['gwsa_mm'].dropna()

# Standard MK
mk_std = mk.original_test(series)
# Modified MK (accounts for autocorrelation)
mk_mod = mk.hamed_rao_modification_test(series)
# Seasonal MK
mk_sea = mk.seasonal_test(series, period=12)
# Sen's slope (mm/month)
sens_month = mk_std.slope
sens_year  = sens_month * 12  # mm/year

print('─' * 55)
print('Mann-Kendall Trend Test — Coastal Ghana GWSA')
print('─' * 55)
for label, res in [
    ('Standard MK',      mk_std),
    ('Modified MK (H&R)',mk_mod),
    ('Seasonal MK',      mk_sea),
]:
    print(f'{label:25s}  trend={res.trend:10s}  p={res.p:.4f}  τ={res.Tau:.3f}')
print()
print(f"Sen's slope:  {sens_month:.4f} mm/month  ({sens_year:.3f} mm/year)")
print(f"Intercept:    {mk_std.intercept:.2f} mm")
print()

# Over GRACE era only (2002–2023)
grace_era = series[(series.index >= '2002-01-01') & (series.index <= '2023-12-31')]
mk_grace  = mk.original_test(grace_era)
print(f'GRACE era (2002–2023) standard MK:  '
      f'trend={mk_grace.trend}  p={mk_grace.p:.4f}  '
      f"slope={mk_grace.slope*12:.3f} mm/yr")
print('─' * 55)


In [ ]:
# ── 4. Drought episode identification ─────────────────────────────────────
# Reclassify SGWA into broader drought / non-drought categories
df['drought_severity'] = pd.cut(df['sgwa'], bins=DROUGHT_BINS,
                                labels=DROUGHT_LABELS, right=True)

# Identify continuous drought episodes (SGWA < -0.5 for ≥ 3 consecutive months)
drought_mask = df['sgwa'] < -0.5
episode_id   = (drought_mask != drought_mask.shift()).cumsum()
episodes = []
for eid, grp in df[drought_mask].groupby(episode_id[drought_mask]):
    if len(grp) >= 3:
        duration = len(grp)
        min_sgwa = grp['sgwa'].min()
        peak_dt  = grp['sgwa'].idxmin().strftime('%Y-%m')
        cat      = grp.loc[grp['sgwa'].idxmin(), 'drought_severity']
        episodes.append({
            'episode_id':  eid,
            'start':       grp.index[0].strftime('%Y-%m'),
            'end':         grp.index[-1].strftime('%Y-%m'),
            'duration_mo': duration,
            'min_sgwa':    round(min_sgwa, 3),
            'peak_month':  peak_dt,
            'category':    str(cat),
        })

df_ep = pd.DataFrame(episodes)
print(f'Drought episodes (SGWA < −0.5, ≥ 3 months): {len(df_ep)}')
print(df_ep.to_string(index=False))
df_ep.to_csv(TAB_DIR / 'drought_episodes.csv', index=False)
print('\nSaved:', TAB_DIR / 'drought_episodes.csv')


In [ ]:
# ── 5. CHIRPS rainfall anomaly ────────────────────────────────────────────
print('Loading CHIRPS ...')

# CHIRPS uses lat -50..50, lon -180..180
ds_ch = xr.open_dataset(CHIRPS_FILE, engine='netcdf4')
# Detect variable name
precip_var = [v for v in ds_ch.data_vars if 'precip' in v.lower() or 'rain' in v.lower()][0]
print(f'  variable: {precip_var}')

# Select Ghana bbox — CHIRPS lat may be descending
lat_vals = ds_ch.latitude.values
lon_vals = ds_ch.longitude.values
lat_s = slice(4.0, 11.5) if lat_vals[0] < lat_vals[-1] else slice(11.5, 4.0)

rain_gh = ds_ch[precip_var].sel(latitude=lat_s, longitude=slice(-3.5, 1.5))
rain_ts = rain_gh.mean(dim=['latitude','longitude'], skipna=True).to_series()
rain_ts.index = pd.DatetimeIndex(rain_ts.index).to_period('M').to_timestamp()
rain_ts = rain_ts[~rain_ts.index.duplicated(keep='first')].sort_index()
rain_ts = rain_ts[(rain_ts.index >= '1981-01-01')]
ds_ch.close()

# Baseline 1981–2010
RAIN_BASELINE = slice('1981-01-01', '2010-12-31')
base_clim = rain_ts.loc[RAIN_BASELINE].groupby(rain_ts.loc[RAIN_BASELINE].index.month).mean()
rain_anom = rain_ts - rain_ts.index.to_series().apply(lambda d: base_clim.get(d.month, np.nan))
rain_anom.name = 'rain_anom_mm'

# Save
rain_anom.to_csv(OUT_TS / 'chirps_rainfall_anomaly_monthly.csv', header=True)
print(f'CHIRPS: {len(rain_ts)} months  '
      f'({rain_ts.index[0].strftime("%Y-%m")} → {rain_ts.index[-1].strftime("%Y-%m")})')
print(f'Annual mean rainfall: {rain_ts.groupby(rain_ts.index.year).sum().mean():.1f} mm/yr')
print('Saved CHIRPS anomaly CSV')


In [ ]:
# ── 6. GWSA–rainfall correlation ──────────────────────────────────────────
# Align on period to avoid start-vs-end-of-month timestamp mismatch
gwsa_aligned = df['gwsa_mm'].dropna().copy()
gwsa_aligned.index = gwsa_aligned.index.to_period('M')
rain_aligned = rain_anom.dropna().copy()
rain_aligned.index = rain_aligned.index.to_period('M')
common = gwsa_aligned.index.intersection(rain_aligned.index)
gwsa_c = gwsa_aligned.loc[common]
rain_c = rain_aligned.loc[common]

r, p_val = stats.pearsonr(rain_c, gwsa_c)
lag_cors = {lag: gwsa_c.corr(rain_c.shift(lag)) for lag in range(0, 13)}

print(f'Pearson r (GWSA vs rainfall anomaly, lag-0): {r:.3f}  p={p_val:.4f}')
print()
print('Lagged correlation (GWSA vs rainfall at lag months):')
for lag, cor in lag_cors.items():
    bar = '█' * int(abs(cor) * 30)
    sign = '+' if cor >= 0 else '-'
    print(f'  lag {lag:2d}: {sign}{bar} {cor:.3f}')

best_lag = max(lag_cors, key=lambda k: abs(lag_cors[k]))
print(f'\nBest lag: {best_lag} months  (r={lag_cors[best_lag]:.3f})')


In [ ]:
# ── 7. Annual cycle and seasonal anomaly ──────────────────────────────────
# Use only the GRACE era (2002–2023) for consistency with gridded data
era = df.loc['2002-01-01':'2023-12-31', 'gwsa_mm'].dropna()

seasonal_mean = era.groupby(era.index.month).mean()
print('Mean GWSA by calendar month (mm EWH, 2002–2023 baseline):')
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
for m, mn in zip(range(1,13), month_names):
    if m in seasonal_mean.index:
        print(f'  {mn}: {seasonal_mean[m]:+7.1f} mm')


In [ ]:
# ── 8. Pixel-wise Mann-Kendall trend from gridded GWSA ────────────────────
ds_gw = xr.open_dataset(OUT_GW / 'gwsa_monthly_grid.nc')
gwsa_g = ds_gw['gwsa']

lats = gwsa_g.lat.values
lons = gwsa_g.lon.values
n_lat, n_lon = len(lats), len(lons)

slope_arr  = np.full((n_lat, n_lon), np.nan)
pval_arr   = np.full((n_lat, n_lon), np.nan)
trend_arr  = np.full((n_lat, n_lon), np.nan)  # +1 increasing, -1 decreasing, 0 no-trend

print(f'Running pixel-wise MK on {n_lat}×{n_lon} grid ...')
for i, lat in enumerate(lats):
    for j, lon in enumerate(lons):
        ts = gwsa_g.sel(lat=lat, lon=lon).values
        valid = ~np.isnan(ts)
        if valid.sum() < 12:
            continue
        res = mk.original_test(ts[valid])
        slope_arr[i, j]  = res.slope * 12        # mm/year
        pval_arr[i, j]   = res.p
        trend_arr[i, j]  = 1 if res.trend == 'increasing' else (
                           -1 if res.trend == 'decreasing' else 0)

ds_gw.close()

# Save as NetCDF
trend_ds = xr.Dataset(
    {
        'sens_slope_mm_yr': xr.DataArray(slope_arr,  dims=['lat','lon'],
            coords={'lat': lats, 'lon': lons},
            attrs={'units': 'mm/year', 'long_name': "Sen's slope of GWSA"}),
        'mk_pvalue':        xr.DataArray(pval_arr,   dims=['lat','lon'],
            coords={'lat': lats, 'lon': lons},
            attrs={'long_name': 'Mann-Kendall p-value'}),
        'mk_trend':         xr.DataArray(trend_arr,  dims=['lat','lon'],
            coords={'lat': lats, 'lon': lons},
            attrs={'long_name': 'Trend direction: 1=inc, -1=dec, 0=none'}),
    },
    attrs={'title': 'Pixel-wise Mann-Kendall trend — coastal Ghana GWSA 2002–2023'}
)
out_trend = OUT_GW / 'gwsa_trend_pixelwise.nc'
trend_ds.to_netcdf(out_trend)
print('Saved:', out_trend)

sig_mask = pval_arr < 0.05
print(f'Pixels with significant trend (p<0.05): {sig_mask.sum()} / {n_lat*n_lon}')
neg_sig  = ((slope_arr < 0) & sig_mask).sum()
pos_sig  = ((slope_arr > 0) & sig_mask).sum()
print(f'  Significant decline:  {neg_sig}')
print(f'  Significant increase: {pos_sig}')
if np.any(~np.isnan(slope_arr)):
    print(f'  Slope range: {np.nanmin(slope_arr):.2f} to {np.nanmax(slope_arr):.2f} mm/yr')


In [ ]:
# ── 9. GDSI — gridded drought frequency map ───────────────────────────────
ds_gw = xr.open_dataset(OUT_GW / 'gwsa_monthly_grid.nc')
gwsa_g = ds_gw['gwsa']

# Standardise each pixel independently
gwsa_mean = gwsa_g.mean('time', skipna=True)
gwsa_std  = gwsa_g.std('time', skipna=True)
sgwa_g    = (gwsa_g - gwsa_mean) / gwsa_std.where(gwsa_std > 0)

# Drought frequency = fraction of months with SGWA < -1.0 (moderate or worse)
drought_freq_g = (sgwa_g < -1.0).sum('time', skipna=True) / sgwa_g.count('time') * 100
drought_freq_g.name = 'drought_freq_pct'
drought_freq_g.attrs = {
    'units': '%',
    'long_name': 'Fraction of months with SGWA < -1.0 (moderate+ drought)',
}

# Mean SGWA for most recent 5 years (2019–2023)
recent = sgwa_g.sel(time=slice('2019-01-01','2023-12-31'))
recent_mean = recent.mean('time', skipna=True)
recent_mean.name = 'sgwa_recent_mean'

ds_gw.close()

# Save
gdsi_ds = xr.Dataset({'drought_freq_pct': drought_freq_g,
                       'sgwa_recent_mean': recent_mean})
gdsi_ds.to_netcdf(OUT_GW / 'gdsi_drought_metrics_grid.nc')
print('Saved:', OUT_GW / 'gdsi_drought_metrics_grid.nc')
print('Drought frequency (% months in moderate+ drought):')
print(drought_freq_g.values.round(1))


In [ ]:
# ── 10. Figure: GWSA time series with trend line and drought episodes ──────
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Coastal Ghana Groundwater Storage Anomaly — Trend and Drought Analysis',
             fontsize=13, fontweight='bold', y=0.98)

era_idx = df.index[(df.index >= '2002-01-01') & (df.index < '2027-01-01')]
era_df  = df.loc[era_idx]

# ── Panel A: GWSA with trend line ──────────────────────────────────────────
ax = axes[0]
pos = era_df['gwsa_mm'].clip(lower=0)
neg = era_df['gwsa_mm'].clip(upper=0)
ax.bar(era_df.index, pos, color='#2171b5', width=25, alpha=0.85, label='Positive anomaly')
ax.bar(era_df.index, neg, color='#cb181d', width=25, alpha=0.85, label='Negative anomaly')

# GRACE mission gap shading
ax.axvspan(pd.Timestamp('2017-07-01'), pd.Timestamp('2018-06-30'),
           color='#bdbdbd', alpha=0.4, label='GRACE gap (2017–2018)')

# Trend line
x_num = np.arange(len(era_df))
valid_mask = ~era_df['gwsa_mm'].isna()
if valid_mask.sum() > 2:
    slope_mo, intercept, *_ = stats.linregress(x_num[valid_mask],
                                               era_df['gwsa_mm'].values[valid_mask])
    trend_line = slope_mo * x_num + intercept
    ax.plot(era_df.index, trend_line, 'k--', lw=1.8,
            label=f"Trend: {slope_mo*12:.2f} mm/yr")

ax.axhline(0, color='k', lw=0.6, ls='-')
ax.set_ylabel('GWSA (mm EWH)', fontsize=10)
ax.set_title('A  Groundwater Storage Anomaly', loc='left', fontsize=10)
ax.legend(fontsize=8, ncol=4, loc='lower left')
ax.grid(axis='y', alpha=0.3)

# ── Panel B: SGWA with drought classification ─────────────────────────────
ax = axes[1]
colours = {
    'Extreme drought': '#67000d', 'Severe drought': '#cb181d',
    'Moderate drought': '#fb6a4a', 'Mild drought': '#fcbba1',
    'Near normal': '#f0f0f0', 'Mildly wet': '#9ecae1', 'Wet': '#2171b5',
}
for idx_dt, row in era_df.iterrows():
    cat = str(row['drought_severity']) if 'drought_severity' in era_df.columns else 'Near normal'
    c   = colours.get(cat, '#f0f0f0')
    ax.bar(idx_dt, row['sgwa'] if not np.isnan(row['sgwa']) else 0,
           color=c, width=25, alpha=0.9)

ax.axhline(-1.0, color='#cb181d', lw=0.8, ls=':', label='Moderate drought (SGWA=−1)')
ax.axhline(-2.0, color='#67000d', lw=0.8, ls=':', label='Extreme drought (SGWA=−2)')
ax.axhline(0,    color='k', lw=0.6)
legend_patches = [Patch(facecolor=colours[l], label=l) for l in list(colours.keys())[:6]]
ax.legend(handles=legend_patches, fontsize=7, ncol=3, loc='lower left')
ax.set_ylabel('SGWA (standardised)', fontsize=10)
ax.set_title('B  Standardised Groundwater Anomaly (SGWA) — Drought Classification',
             loc='left', fontsize=10)
ax.grid(axis='y', alpha=0.3)

# ── Panel C: CHIRPS rainfall anomaly ──────────────────────────────────────
ax = axes[2]
rain_era = rain_anom.reindex(era_df.index)
pos_r = rain_era.clip(lower=0)
neg_r = rain_era.clip(upper=0)
ax.bar(era_df.index,
       pos_r, color='#2171b5', width=25, alpha=0.7, label='Above-normal rainfall')
ax.bar(era_df.index, neg_r, color='#fc8d59', width=25, alpha=0.7,
       label='Below-normal rainfall')
ax.axhline(0, color='k', lw=0.6)
ax.set_ylabel('Rainfall anomaly (mm/mo)', fontsize=10)
ax.set_title('C  CHIRPS Rainfall Anomaly (vs 1981–2010 baseline)',
             loc='left', fontsize=10)
ax.legend(fontsize=8, loc='lower left')
ax.grid(axis='y', alpha=0.3)

fig.tight_layout(rect=[0, 0, 1, 0.97])
out_fig = FIG_DIR / 'gwsa_trend_drought_timeseries.png'
fig.savefig(out_fig, dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', out_fig)


In [ ]:
# ── 11. Figure: seasonal GWSA cycle + rainfall ────────────────────────────
era_gwsa = df.loc['2002-01-01':'2023-12-31', 'gwsa_mm'].dropna()
era_rain = rain_ts.loc['2002-01-01':'2023-12-31']

gwsa_seas = era_gwsa.groupby(era_gwsa.index.month).agg(['mean','std'])
rain_seas = era_rain.groupby(era_rain.index.month).mean()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
x = np.arange(1, 13)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bars = ax2.bar(x, rain_seas.values, color='#9ecae1', alpha=0.6,
               width=0.6, label='Mean rainfall')
ax1.errorbar(x, gwsa_seas['mean'].values, yerr=gwsa_seas['std'].values,
             fmt='-o', color='#2171b5', lw=2, capsize=4, markersize=6,
             label='Mean GWSA ± 1 SD')
ax1.axhline(0, color='k', lw=0.6)

ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('GWSA (mm EWH)', fontsize=11, color='#2171b5')
ax2.set_ylabel('Rainfall (mm/month)', fontsize=11, color='#4292c6')
ax1.set_xticks(x)
ax1.set_xticklabels(month_labels)
ax1.tick_params(axis='y', labelcolor='#2171b5')
ax2.tick_params(axis='y', labelcolor='#4292c6')
ax1.set_title('Seasonal Cycle of GWSA and Rainfall — Coastal Ghana (2002–2023)',
              fontsize=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

fig.tight_layout()
fig.savefig(FIG_DIR / 'gwsa_seasonal_cycle.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'gwsa_seasonal_cycle.png')


In [ ]:
# ── 12. Figure: pixel-wise trend map ──────────────────────────────────────
import cartopy.crs as ccrs
import cartopy.feature as cfeature

trend_nc = xr.open_dataset(OUT_GW / 'gwsa_trend_pixelwise.nc')
slope    = trend_nc['sens_slope_mm_yr']
pval     = trend_nc['mk_pvalue']

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection=ccrs.PlateCarree())

vmax = max(abs(float(slope.min())), abs(float(slope.max())))
if vmax == 0:
    vmax = 1.0
vmax = min(vmax, 30)

cmap = plt.cm.RdBu
cf = ax.pcolormesh(slope.lon, slope.lat, slope.values,
                   cmap=cmap, vmin=-vmax, vmax=vmax,
                   transform=ccrs.PlateCarree())
# Hatching for significant pixels (p < 0.05)
sig = (pval.values < 0.05)
for i, lat in enumerate(slope.lat.values):
    for j, lon in enumerate(slope.lon.values):
        if sig[i, j]:
            ax.plot(lon, lat, 'k.', markersize=8, transform=ccrs.PlateCarree())

ax.add_feature(cfeature.COASTLINE, lw=0.8)
ax.add_feature(cfeature.BORDERS, lw=0.5, linestyle=':')
ax.add_feature(cfeature.LAND, facecolor='#f5f5f5', zorder=0)

cbar = fig.colorbar(cf, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Sen's slope (mm/year)", fontsize=10)
ax.set_extent([-3.5, 1.5, 4.0, 11.5], crs=ccrs.PlateCarree())
ax.set_title("GWSA Trend Map — Coastal Ghana (2002–2023)\n"
             "Dots = statistically significant (p < 0.05)", fontsize=11)
gl = ax.gridlines(draw_labels=True, lw=0.3, alpha=0.5)
gl.top_labels = gl.right_labels = False

trend_nc.close()
fig.tight_layout()
fig.savefig(FIG_DIR / 'gwsa_trend_map.png', dpi=180, bbox_inches='tight')
plt.close(fig)
print('Saved:', FIG_DIR / 'gwsa_trend_map.png')


In [ ]:
# ── 13. Trend summary table ───────────────────────────────────────────────
summary = {
    'Analysis period':       f'{series.index[0].strftime("%Y-%m")} – {series.index[-1].strftime("%Y-%m")}',
    'n_months':              len(series),
    'Mean GWSA (mm)':        round(series.mean(), 2),
    'Std GWSA (mm)':         round(series.std(), 2),
    'Min GWSA (mm)':         round(series.min(), 2),
    'Max GWSA (mm)':         round(series.max(), 2),
    'MK trend':              mk_std.trend,
    'MK p-value':            round(mk_std.p, 4),
    "Sen's slope (mm/yr)":   round(sens_year, 3),
    'Modified MK trend':     mk_mod.trend,
    'Modified MK p-value':   round(mk_mod.p, 4),
    'Seasonal MK trend':     mk_sea.trend,
    'Seasonal MK p-value':   round(mk_sea.p, 4),
    'Drought episodes (≥3mo)': len(df_ep),
    'Best lag GWSA-rain (mo)': best_lag,
    'r GWSA-rain (lag-0)':   round(r, 3),
}

df_sum = pd.DataFrame.from_dict(summary, orient='index', columns=['Value'])
df_sum.index.name = 'Metric'
print(df_sum.to_string())
df_sum.to_csv(TAB_DIR / 'gwsa_trend_summary.csv')
print('\nSaved:', TAB_DIR / 'gwsa_trend_summary.csv')

print()
print('=' * 55)
print('Stage 6 complete — outputs:')
for p in [
    OUT_TS  / 'chirps_rainfall_anomaly_monthly.csv',
    OUT_GW  / 'gwsa_trend_pixelwise.nc',
    OUT_GW  / 'gdsi_drought_metrics_grid.nc',
    TAB_DIR / 'drought_episodes.csv',
    TAB_DIR / 'gwsa_trend_summary.csv',
    FIG_DIR / 'gwsa_trend_drought_timeseries.png',
    FIG_DIR / 'gwsa_seasonal_cycle.png',
    FIG_DIR / 'gwsa_trend_map.png',
]:
    tick = '✓' if Path(p).exists() else '✗ MISSING'
    print(f'  {tick}  {Path(p).name}')
